In [ ]:
# Install the library (and its dependencies) in editable mode from the repository root
%pip install -e ..

In [ ]:
import sys
import logging
from pathlib import Path

# scripts/ holds the thesis-specific visualizers imported dynamically below
scripts_path = str(Path("../scripts").resolve())
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

from sdt_netval import load_network

# Load the network from the SQLite database (Tomasevic benchmark run 01)
db_path = Path("../data/00_raw/01_legacy_tomasevic/benchmark_runs/run01.sqlite").resolve()
G = load_network(db_path)

In [ ]:
import pandas as pd
from sdt_netval import GraphMetrics

report = GraphMetrics(G).generate_full_report()

pd.Series(report).rename("value").to_frame()

In [ ]:
from sdt_netval.pipeline import StageAValidator

data_dir = Path("../data/00_raw/01_legacy_tomasevic/benchmark_runs").resolve()

validator = StageAValidator(data_dir)
validator.process_runs()
validator.save_raw_results(Path("../data/01_processed/01_legacy_tomasevic/stage_a_raw.csv").resolve())

report = validator.full_stability_report()
print("\n=== Topological Stability (Stage A — Legacy Tomašević) ===\n")
print(report["stability"])

In [ ]:
from importlib import import_module

stage_a_viz = import_module("02_stage_a_visualization")
StageAVisualizer = stage_a_viz.StageAVisualizer

viz = StageAVisualizer("../data/01_processed/01_legacy_tomasevic/stage_a_raw.csv")
viz.plot_stability_distributions("../data/01_processed/01_legacy_tomasevic/stage_a_stability.png")
print(viz.plot_summary_statistics())

In [ ]:
from sdt_netval.pipeline import StageBAnalyzer

analyzer = StageBAnalyzer(Path("../data/00_raw/01_legacy_tomasevic/sensitivity_runs").resolve())
analyzer.process_all_runs()
analyzer.save_raw_results(Path("../data/01_processed/01_legacy_tomasevic/stage_b_raw.csv").resolve())
analyzer.save_aggregated_results(Path("../data/01_processed/01_legacy_tomasevic/stage_b_aggregated.csv").resolve())

report = analyzer.full_sensitivity_report()
print(report["aggregated"])

In [ ]:
from importlib import import_module

stage_b_viz = import_module("04_stage_b_visualization")
StageBVisualizer = stage_b_viz.StageBVisualizer

viz = StageBVisualizer("../data/01_processed/01_legacy_tomasevic/stage_b_raw.csv")
viz.plot_modularity_comparison("../data/01_processed/01_legacy_tomasevic/stage_b_modularity_comparison.png")
print(viz.get_summary_statistics())

In [ ]:
import pandas as pd
from sdt_netval import compare_to_baseline

raw = pd.read_csv("../data/01_processed/01_legacy_tomasevic/stage_b_raw.csv")

# Mann-Whitney U of each condition vs the baseline c0, Holm-corrected within each metric
results_df = compare_to_baseline(
    raw, "c0",
    metrics=["alpha_in_degree", "modularity", "average_clustering"],
    conditions=["c1", "c3", "c4", "c8"],
)
results_df.to_csv("../data/01_processed/01_legacy_tomasevic/stage_b_pvalues.csv", index=False)

print(results_df.round(4).to_string(index=False))
print("Significant after correction:", results_df.loc[results_df["significance"] != "ns", ["metric", "condition"]].values.tolist())